# Liberman synapse comparison — true-noise mirror

Oracle **`noise_cat`** conditioning (no Stage 1). Configuration panel **T1, T3, T4, T5** (RF + XGB each) and **L7–L8** (OLS). **No T6** (T5 true ≈ prior T6 oracle).

Run **before** synthesis true-noise CV only if you need Colab NN rows; synthesis HP uses `abr_stage2_hp_tuning_true_noise.ipynb`.

In [1]:
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display

import utils.liberman_classical as lc
import utils.nn_stage2_data as nn2d

importlib.reload(nn2d)
importlib.reload(lc)

from utils.liberman_classical import (
    COMPARISON_PAIRS_TRUE,
    TRUE_NOISE_TREE_CONFIGS,
    animal_noise_series,
    attach_animal_noise_cat,
    derive_comparisons,
    export_artifacts,
    liberman_feature_lists,
    run_config_panel,
)
from utils.nn_colab_export import (
    DEFAULT_TRUE_OUT,
    export_liberman_nn_colab_pack,
)
from utils.nn_colab_train import run_liberman_nn_hp_comparison
from utils.nn_stage2_data import load_nn_stage2_data, splits_for_long_stage2

In [2]:
data = load_nn_stage2_data(join_io_features=False)
sp = splits_for_long_stage2(data)
an = animal_noise_series(data.orig_lib)
lib_tr = attach_animal_noise_cat(sp["lib_train"].copy(), an)
lib_te = attach_animal_noise_cat(sp["lib_test"].copy(), an)
lib_long_tr = attach_animal_noise_cat(sp["lib_long_train"].copy(), an)
lib_long_te = attach_animal_noise_cat(sp["lib_long_test"].copy(), an)
feats = liberman_feature_lists(data.reformatted_orig, data.common_cols)
assert lib_tr["noise_cat"].isin([0, 1]).all()

In [3]:
results = run_config_panel(
    lib_tr,
    lib_te,
    lib_long_tr,
    lib_long_te,
    feats,
    tree_configs=TRUE_NOISE_TREE_CONFIGS,
    skip_stage1=True,
    ols_noise_mode="true",
    verbose=True,
)
assert len(results) == 10
display(results.sort_values(["config_id", "model"]))


=== T1 RF (long, noise=none) ===
  [T1-RF]  S1 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse reg — CV R²: 0.220, test R² (animal×freq): 0.317, RMSE: 3.030

=== T1 XGB (long, noise=none) ===
  [T1-XGB]  S1/S2 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse XGB — CV R²: 0.220, test R² (animal×freq): 0.331, RMSE: 2.998

=== T3 RF (long, noise=true) ===
  [T3-RF]  S1 wide train: 491 | long train: 9751 rows
           RF  — val acc (row): 0.702 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal): 0.952, AUC: 0.909
           Synapse reg — CV R²: 0.453, test R² (animal×freq): 0.611, RMSE: 2.287

=== T3 XGB (long, noise=true) ===
  [T3-XGB]  S1/S2 wide train: 

,config_id,model,format,noise_label,r2_test,rmse_test
8,L7,OLS,wide,none,0.228920,3.090075
9,L8,OLS,wide,true,0.461984,2.581172
0,T1,RF,long,none,0.316922,3.030018
1,T1,XGB,long,none,0.331368,2.997807
2,T3,RF,long,true,0.610980,2.286631
3,T3,XGB,long,true,0.556463,2.441602
4,T4,RF,wide,none,0.486302,2.522162
5,T4,XGB,wide,none,0.469613,2.562805
6,T5,RF,wide,true,0.663359,2.041750
7,T5,XGB,wide,true,0.664509,2.038262


In [4]:
comparisons = derive_comparisons(
    results,
    wide_train=lib_tr,
    wide_test=lib_te,
    long_train=lib_long_tr,
    long_test=lib_long_te,
    feats=feats,
    comparison_pairs=COMPARISON_PAIRS_TRUE,
    tree_configs=TRUE_NOISE_TREE_CONFIGS,
    skip_stage1=True,
)
display(comparisons)

,question,model,config_a,config_b,r2_a,r2_b,delta_r2,rmse_a,rmse_b,delta_rmse,...,mean_sq_err_b,mean_sq_err_diff_a_minus_b,f_stat,f_pvalue,variances_unequal,t_test,t_stat,t_pvalue,significant_at_alpha,alpha
0,Q3_format,RF,T1,T4,0.316922,0.486302,0.169380,3.030018,2.522162,-0.507856,...,6.361302,2.819707,2.157682,0.000024,True,welch,1.710541,0.088584,False,0.05
1,Q1_noise,RF,T4,T5,0.486302,0.663359,0.177057,2.522162,2.041750,-0.480412,...,4.168745,2.192557,1.666232,0.004775,True,welch,1.868459,0.062951,False,0.05
2,Q1_noise_long,RF,T1,T3,0.316922,0.610980,0.294058,3.030018,2.286631,-0.743387,...,5.228681,3.952328,2.023482,0.000106,True,welch,2.372843,0.018503,True,0.05
3,Q3_format,XGB,T1,T4,0.331368,0.469613,0.138246,2.997807,2.562805,-0.435003,...,6.567969,2.418881,1.905663,0.000381,True,welch,1.474979,0.141610,False,0.05
4,Q1_noise,XGB,T4,T5,0.469613,0.664509,0.194895,2.562805,2.038262,-0.524543,...,4.154511,2.413457,1.453458,0.038279,True,welch,1.930837,0.054682,False,0.05
5,Q1_noise_long,XGB,T1,T3,0.331368,0.556463,0.225095,2.997807,2.441602,-0.556205,...,5.961422,3.025428,1.583563,0.010986,True,welch,1.783470,0.075795,False,0.05


## Colab pack + NN HP (true noise)

In [5]:
colab_pack_dir = export_liberman_nn_colab_pack(
    DEFAULT_TRUE_OUT, noise_label="true"
)
print(f"Upload to Colab: {colab_pack_dir.resolve()}")
cache_dir = Path("figures/cache")
nn_out = DEFAULT_TRUE_OUT / "results"
summary, nn_results = run_liberman_nn_hp_comparison(
    DEFAULT_TRUE_OUT, nn_out, verbose=True
)
display(nn_results.sort_values("r2_test", ascending=False).head())
nn_results.to_parquet(
    cache_dir / "liberman_nn_hp_comparison_true_noise.parquet", index=False
)

Exported Colab pack → /Users/nowaki027/MSDS/Practicum/figures/cache/nn_colab_liberman_true
  train 7869 rows, 68 animals | validate 1882 | test 2436
  tabular dim=14, wave_full_len=201
Upload to Colab: /Users/nowaki027/MSDS/Practicum/figures/cache/nn_colab_liberman_true

=== HP tuning: mlp ===
  [mlp 1/72] hidden=(64,) dropout=0.1 lr=0.0001 wd=0.0001 rmse=6.0388
  [mlp 2/72] hidden=(64,) dropout=0.1 lr=0.0001 wd=0.001 rmse=6.0395
  [mlp 3/72] hidden=(64,) dropout=0.1 lr=0.001 wd=0.0001 rmse=2.5079
  [mlp 4/72] hidden=(64,) dropout=0.1 lr=0.001 wd=0.001 rmse=2.5097
  [mlp 5/72] hidden=(64,) dropout=0.1 lr=0.005 wd=0.0001 rmse=2.4289
  [mlp 6/72] hidden=(64,) dropout=0.1 lr=0.005 wd=0.001 rmse=2.4309
  [mlp 7/72] hidden=(64,) dropout=0.2 lr=0.0001 wd=0.0001 rmse=6.0623
  [mlp 8/72] hidden=(64,) dropout=0.2 lr=0.0001 wd=0.001 rmse=6.0631
  [mlp 9/72] hidden=(64,) dropout=0.2 lr=0.001 wd=0.0001 rmse=2.5269
  [mlp 10/72] hidden=(64,) dropout=0.2 lr=0.001 wd=0.001 rmse=2.5268
  [mlp 11/72] h

,config_id,model,format,noise_label,r2_test,rmse_test
0,N2,cnn,long,true,0.617830,2.266409
2,N1,mlp,long,true,0.551933,2.454039
1,N3,cnn_full,long,true,0.503734,2.582659


In [6]:
cache_dir = Path("figures/cache")
pq_path, json_path, comp_path = export_artifacts(
    results, comparisons, cache_dir, stem="liberman_classical_true_noise"
)
print(pq_path, json_path, comp_path)

figures/cache/liberman_classical_true_noise_comparison.parquet figures/cache/liberman_true_noise_best_tree_config.json figures/cache/liberman_classical_true_noise_comparisons.parquet
